# Consulta — Índices EEG adicionales para clasificación en BCI

## Introducción general

Las señales EEG basadas en Motor Imagery (MI) proporcionan interacción y comunicación entre pacientes con parálisis y el mundo exterior para mover y controlar dispositivos externos como sillas de ruedas y cursores. Sin embargo, los enfoques actuales en el diseño de sistemas MI-BCI requieren métodos efectivos de extracción de características y algoritmos de clasificación para adquirir características discriminativas de las señales EEG, debido a su naturaleza no lineal y no estacionaria[1].

Los métodos de extracción de características tradicionales en BCI se clasifican en dominio temporal, frecuencial, espacial y tiempo-frecuencial. Adicionalmente, métodos como el análisis de redes complejas y las características basadas en entropía ofrecen nuevas perspectivas para analizar las señales EEG que complementan la densidad espectral de potencia.

## Índice 1 — ERD/ERS (Event-Related Desynchronization / Synchronization)

### Descripción de la medida

La imaginación de un movimiento motor elicita una atenuación de corta duración de los ritmos en las bandas de frecuencia alpha (8-13 Hz) y beta (13-25 Hz), llamada desincronización relacionada con eventos (ERD). Esta ERD es similar a la actividad cortical durante la ejecución real del movimiento, lo que hace que el motor imagery sea adecuado para aplicaciones BCI en rehabilitación de pacientes con déficits motores[2].

El ERD se define como una disminución de la potencia del EEG relativa a un estado de reposo precedente dentro de las bandas mu (8-13 Hz) y beta (14-30 Hz), correlacionada con la ejecución motora, el motor imagery y la observación motora. La ERS es un rebote de la potencia en la banda beta después de la terminación del movimiento o la imaginación[3].

### Fórmula

$$ERD/ERS(\%) = \frac{P_{tarea} - P_{reposo}}{P_{reposo}} \times 100$$

Donde $P_{tarea}$ es la potencia durante la tarea y $P_{reposo}$ es la potencia durante el reposo (línea base). Valores negativos indican ERD (desincronización) y valores positivos indican ERS (sincronización).

### Relevancia para la clasificación en BCI

La ERD durante el motor imagery de muñeca se asocia con amplitudes de MEP significativamente aumentadas y SICI reducida, lo que demuestra que la magnitud de la ERD representa la excitabilidad de M1. Este estudio proporciona evidencia electrofisiológica de que una tarea de motor imagery que involucra ERD puede inducir cambios en la excitabilidad corticoespinal similares a los que acompañan a los movimientos reales[4].

Las BCI basadas en motor imagery explotan la modulación de los ritmos sensoriomotores sobre las cortezas motoras, conocidos como ERD y ERS. La interpretación del ERD/ERS está directamente relacionada con la selección de la línea base utilizada para estimarlos, lo cual puede resultar en una visualización engañosa si no se realiza correctamente[5].

### Librerías utilizadas

`scipy.signal` y `numpy` 

### Uso en los datos del proyecto

In [10]:
from scipy import signal
import numpy as np

def calcular_erd_ers(raw_segmento, raw_reposo, fs, banda):
    """
    Calcula ERD/ERS para un segmento de tarea respecto al reposo.
    
    Parámetros:
        raw_segmento: array (n_canales, n_muestras) — segmento T1 o T2
        raw_reposo:   array (n_canales, n_muestras) — segmento T0
        fs: frecuencia de muestreo (160 Hz)
        banda: tupla (f_min, f_max) — ej: (8, 13) para alpha
    
    Retorna:
        erd_ers: array (n_canales,) con el porcentaje ERD/ERS por canal
    """
    f_min, f_max = banda
    resultado = {}
    
    for idx in range(raw_segmento.shape[0]):
        # PSD de la tarea
        freqs, psd_tarea = signal.welch(
            raw_segmento[idx], fs=fs, nperseg=int(fs*2))
        # PSD del reposo (línea base)
        freqs, psd_reposo = signal.welch(
            raw_reposo[idx], fs=fs, nperseg=int(fs*2))
        
        # Índices de la banda de interés
        idx_banda = np.where((freqs >= f_min) & (freqs <= f_max))[0]
        
        # Potencia promedio en la banda
        p_tarea = np.mean(psd_tarea[idx_banda])
        p_reposo = np.mean(psd_reposo[idx_banda])
        
        # ERD/ERS como porcentaje de cambio relativo
        erd_ers = ((p_tarea - p_reposo) / (p_reposo + 1e-10)) * 100
        resultado[idx] = erd_ers
    
    return resultado

# En nuestros datos:
# Se espera ERD negativo en alpha y beta de C3 durante imaginación mano derecha (T2)
# Se espera ERD negativo en alpha y beta de C4 durante imaginación mano izquierda (T1)
# Esto refleja la dominancia contralateral del control motor

## Índice 2 — Parámetros de Hjorth

### Descripción de la medida

Los parámetros de Hjorth describen la señal EEG mediante tres medidas en el dominio temporal. La Actividad mide el cuadrado de la desviación estándar de la amplitud, es decir, la varianza de las señales EEG en una ventana de análisis. La Movilidad se define como la potencia media de la derivada normalizada de las señales EEG. La Complejidad representa la razón entre las movilidades del EEG, relacionando la primera derivada y la señal origina[6].

### Fórmulas

$$Actividad = \sigma^2(x)$$

$$Movilidad = \frac{\sigma(x')}{\sigma(x)}$$

$$Complejidad = \frac{Movilidad(x')}{Movilidad(x)}$$

Donde $x$ es la señal EEG, $x'$ es su primera derivada y $\sigma$ es la desviación estándar.

**Interpretación física:**
- **Actividad** → potencia total de la señal
- **Movilidad** → frecuencia media dominante
- **Complejidad** → qué tan similar es la señal a una sinusoide pura (complejidad = 1 para sinusoide perfecta)

### Relevancia para la clasificación en BCI

Al analizar las señales EEG con el parámetro de Hjorth en sistemas BCI, la precisión de clasificación mejoró en un 4.4% en promedio. Dado que las señales EEG tienen una propiedad no estacionaria y sus características de frecuencia difieren de individuo a individuo, los parámetros de Hjorth permiten seleccionar la banda de frecuencia y el tiempo dominantes para extraer características discriminantes[7].

Un promedio de precisión del 89.7 ± 0.78% fue obtenido al discriminar características de ejecución motora e imaginación motora en los canales C3, Cz y C4 usando parámetros de Hjorth combinados con FastICA y SVM, confirmando su efectividad para diferenciar estados motores en BCI[6]

### Librerías utilizadas

`numpy` 

### Uso en los datos del proyecto

In [11]:
import numpy as np

def calcular_hjorth(señal):
    """
    Calcula los 3 parámetros de Hjorth sobre un segmento EEG.
    
    Parámetros:
        señal: array 1D con el segmento de un canal EEG
    
    Retorna:
        actividad: varianza de la señal (potencia total)
        movilidad: frecuencia media de la señal
        complejidad: similitud con señal sinusoidal pura
    """
    # Actividad: varianza de la señal
    actividad = np.var(señal)
    
    # Primera derivada (diferencias entre muestras consecutivas)
    primera_derivada = np.diff(señal)
    
    # Movilidad: proporción de desviaciones estándar
    movilidad = np.std(primera_derivada) / (np.std(señal) + 1e-10)
    
    # Segunda derivada
    segunda_derivada = np.diff(primera_derivada)
    
    # Complejidad: cambio en la movilidad
    movilidad_deriv = np.std(segunda_derivada) / (np.std(primera_derivada) + 1e-10)
    complejidad = movilidad_deriv / (movilidad + 1e-10)
    
    return actividad, movilidad, complejidad

# En nuestros datos:
# Se aplica sobre segmentos T0 (reposo), T1 (mano izq), T2 (mano der)
# en los 9 canales de la grilla sensoriomotora
# Se espera que la movilidad y complejidad cambien durante motor imagery
# reflejando la reorganización de la actividad cortical

## Índice 3 — Entropía Espectral

### Descripción de la medida

La entropía de información, con sus características no lineales intrínsecas, captura efectivamente el comportamiento dinámico de las señales EEG, abordando las limitaciones de los métodos lineales tradicionales. Entre las 13 medidas de entropía más utilizadas en MI-BCI publicadas entre 2019 y 2023, la entropía de Shannon (13%) y la entropía espectral son de las más frecuentemente empleadas en extracción de características para motor imagery[7].

La entropía espectral mide qué tan uniforme o concentrada está la distribución de potencia en el espectro de frecuencias, basándose en la entropía de Shannon aplicada a la PSD normalizada:

$$H_{espectral} = -\sum_{f} p(f) \cdot \log_2(p(f))$$

Donde $p(f)$ es la PSD normalizada para que sume 1 (distribución de probabilidad):

$$p(f) = \frac{PSD(f)}{\sum_f PSD(f)}$$

**Interpretación física:**
- **Alta entropía** → potencia distribuida en muchas frecuencias → señal compleja y aleatoria
- **Baja entropía** → potencia concentrada en pocas frecuencias → señal más regular y ordenada

### Relevancia para la clasificación en BCI

Los métodos basados en entropía son los más adecuados para la caracterización de señales no estacionarias como el EEG. La entropía espectral combinada con correlación cruzada ha demostrado ser un método robusto de extracción de características para el control de dispositivos robóticos asistivos mediante motor imagery EEG[8].

Las características espectrales de energía, entropía y varianza de las sub-bandas de frecuencia del EEG (δ, θ, α, β y γ) calculadas mediante FFT son ampliamente utilizadas en sistemas BCI, extrayéndose de múltiples canales EEG para mejorar significativamente la clasificación de tareas de motor imagery.

### Librerías utilizadas

`scipy.signal` y `numpy`

### Uso en los datos del proyecto

In [12]:
from scipy import signal
import numpy as np

def calcular_entropia_espectral(señal, fs, f_min, f_max):
    """
    Calcula la entropía espectral de una banda de frecuencia.
    
    Parámetros:
        señal: array 1D con el segmento de un canal EEG
        fs: frecuencia de muestreo (160 Hz en nuestros datos)
        f_min, f_max: límites de la banda (ej: 8, 13 para alpha)
    
    Retorna:
        entropia: valor escalar de entropía espectral en la banda
    """
    # Calcular PSD con Welch (mismo método del proyecto 1)
    freqs, psd = signal.welch(señal, fs=fs, nperseg=int(fs*2))
    
    # Filtrar frecuencias de la banda de interés
    idx_banda = np.where((freqs >= f_min) & (freqs <= f_max))[0]
    psd_banda = psd[idx_banda]
    
    # Normalizar para obtener distribución de probabilidad
    psd_norm = psd_banda / (np.sum(psd_banda) + 1e-10)
    
    # Entropía de Shannon sobre el espectro normalizado
    entropia = -np.sum(psd_norm * np.log2(psd_norm + 1e-10))
    
    return entropia

# En nuestros datos:
# Se calcula por banda (alpha y beta) en los 9 canales sensoriomotores
# para segmentos T0, T1 y T2
# Se espera mayor entropía durante motor imagery que en reposo
# indicando mayor complejidad de la actividad cortical durante imaginación

# Plan de Análisis — Segundo Proyecto BCI

## 1. Contexto y condiciones de estudio

En el proyecto 1 se procesaron 109 sujetos con 1526 archivos EDF, aplicando un pipeline de filtrado (pasa banda 0.5-40 Hz + Notch 60 Hz) sobre una grilla sensoriomotora de 9 canales (Fc3, Fcz, Fc4, C3, Cz, C4, Cp3, Cpz, Cp4)[6][3] y se calculó la PSD por bandas usando el método de Welch.

Para este proyecto, el análisis se enfoca en 3 condiciones específicas extraídas de los segmentos T0, T1 y T2 de los runs de imaginación (R04, R08, R12):

| Condición | Etiqueta | Runs | Descripción |
|---|---|---|---|
| Reposo | T0 | R04, R08, R12 | Sin actividad motora — línea base |
| Imaginación mano derecha | T2 | R04, R08, R12 | Motor imagery mano derecha |
| Imaginación mano izquierda | T1 | R04, R08, R12 | Motor imagery mano izquierda |



## 2. Pipeline de procesamiento (heredado del proyecto 1)

Los 9 canales de la grilla sensoriomotora (Fc3, Fcz, Fc4, C3, Cz, C4, Cp3, Cpz, Cp4) se justifican con base en los resultados del proyecto 1, donde se obtuvo una precisión promedio del 89.7 ± 0.78% al discriminar características de ejecución motora e imaginación motora en los canales C3, Cz y C4, confirmando que estos canales son los más relevantes para motor imagery.Las señales EEG son preprocesadas usando filtros FIR, seguidos de extracción de características multidominio incluyendo energía, patrones espaciales comunes y densidad espectral de potencia, que son fusionadas para mejorar la precisión de clasificación[10].

El procesamiento ya validado en el proyecto 1 se mantiene:

```
Señal EDF cruda (64 canales, 160 Hz, ~125s por archivo)
    ↓
Selección grilla sensoriomotora 3×3
(Fc3, Fcz, Fc4, C3, Cz, C4, Cp3, Cpz, Cp4)
    ↓
Filtro pasa banda FIR (0.5 – 40 Hz)
    ↓
Filtro Notch (60 Hz)
    ↓
Segmentación por anotaciones
T0 = reposo (~4.2s) | T1 = mano izq (~4.1s) | T2 = mano der (~4.1s)
    ↓
Extracción de características por segmento y canal
```



## 3. Extracción de características


Para cada segmento y canal se extraen 4 grupos de características:

### 3.1 PSD por bandas (proyecto 1 — ya calculada)

La PSD calculada con el método de Welch en el proyecto 1 es la base del análisis espectral. Un enfoque bien establecido en BCI basados en motor imagery ha sido usar la PSD de las señales como características para discriminar entre tareas motoras, obteniendo precisiones medias del 90 ± 8% y 87 ± 7% para las bandas mu y beta respectivamente[6].

Potencia promedio en cada banda usando método de Welch con ventana de 2 segundos (resolución espectral 0.5 Hz):

$$PSD_{banda} = \frac{1}{|B|} \sum_{f \in B} S(f)$$

Donde $S(f)$ es la densidad espectral y $B$ es el conjunto de frecuencias de la banda.

**Resultado:** 5 valores por canal → 45 características totales (5 bandas × 9 canales)

### 3.2 ERD/ERS (nuevo índice 1)

La imaginación de un movimiento motor elicita una atenuación de corta duración de los ritmos en las bandas alpha (8-13 Hz) y beta (13-25 Hz), llamada ERD. Esta ERD es similar a la actividad cortical durante la ejecución real del movimiento, lo que hace que el motor imagery sea adecuado para sistemas BCI en rehabilitación[2].

Cuantifica el cambio relativo de potencia durante la imaginación respecto al reposo:

$$ERD/ERS(\%) = \frac{P_{tarea} - P_{reposo}}{P_{reposo}} \times 100$$


**Resultado:** 2 valores por canal → 18 características (2 bandas × 9 canales)

### 3.3 Parámetros de Hjorth (nuevo índice 2)

Al analizar las señales EEG con el parámetro de Hjorth en sistemas BCI, la precisión de clasificación mejora un 4.4% en promedio. Los parámetros permiten seleccionar la banda de frecuencia y el tiempo dominantes para extraer características discriminantes, siendo especialmente útiles dado que las señales EEG tienen propiedades no estacionarias que varían entre individuos[7].

Describen la señal en el dominio temporal:

$$Actividad = \sigma^2(x) \qquad Movilidad = \frac{\sigma(x')}{\sigma(x)} \qquad Complejidad = \frac{Movilidad(x')}{Movilidad(x)}$$

**Resultado:** 3 valores por canal → 27 características (3 parámétricas × 9 canales)

### 3.4 Entropía Espectral (nuevo índice 3)

La entropía de información, con sus características no lineales intrínsecas, captura efectivamente el comportamiento dinámico de las señales EEG, abordando las limitaciones de los métodos lineales tradicionales en la captura de características no lineales. Entre las medidas más utilizadas en MI-BCI entre 2019 y 2023, la entropía de Shannon es de las más frecuentemente empleadas[8].

Mide la complejidad de la distribución espectral:

$$H = -\sum_{f} p(f) \cdot \log_2(p(f))$$

Se calcula por banda de frecuencia.

**Resultado:** 5 valores por canal → 45 características (5 bandas × 9 canales)



## 4. Estructura del DataFrame final

| Columna | Tipo | Ejemplo |
|---|---|---|
| sujeto | str | S001 |
| condicion | str | imaginacion_mano_der |
| psd_alpha_C3 | float | 38.5 µV²/Hz |
| erd_alpha_C3 | float | -23.4% |
| erd_beta_C3 | float | -18.7% |
| hjorth_actividad_C3 | float | 0.0023 |
| hjorth_movilidad_C3 | float | 0.42 |
| hjorth_complejidad_C3 | float | 1.87 |
| entropia_alpha_C3 | float | 3.21 |
| ... | ... | ... |

**Total de características:** ~135 características × 9 canales


## 5. Análisis estadístico

### 5.1 Estadística descriptiva

Se usarán las representaciones recomendadas por data-to-viz.com para comparar las 3 condiciones:

- **Boxplots** → comparar distribuciones de ERD/ERS entre condiciones en C3 y C4
- **Violin plots** → mostrar forma de distribución de Hjorth y entropía
- **Heatmap** → ERD/ERS por canal y condición — permite ver lateralización
- **Barplot con error** → media ± desviación estándar por condición

### 5.2 Prueba de normalidad

Shapiro-Wilk sobre cada índice y condición (máximo 50 muestras):

- $H_0$: Los datos siguen distribución normal
- $H_1$: Los datos **NO** siguen distribución normal

Basado en los resultados del proyecto 1, se espera que los datos no sean normales → pruebas no paramétricas.

### 5.3 Hipótesis

**Hipótesis 1 — ERD en alpha y beta durante motor imagery:**

- $H_0$: No hay diferencia en ERD entre reposo e imaginación de movimiento
- $H_1$: El ERD es significativamente más negativo durante imaginación que en reposo
- **Prueba:** Mann-Whitney U (reposo vs imaginación mano der, reposo vs imaginación mano izq)

**Hipótesis 2 — Lateralización hemisférica C3 vs C4:**

- $H_0$: No hay diferencia entre C3 y C4 durante imaginación de mano derecha vs izquierda
- $H_1$: La imaginación de mano derecha produce mayor ERD en C3 y la de mano izquierda en C4
- **Prueba:** Mann-Whitney U (C3 vs C4 por condición)

**Hipótesis 3 — Diferenciación entre manos:**

- $H_0$: Los índices no permiten discriminar entre imaginación de mano derecha e izquierda
- $H_1$: Existe diferencia significativa en al menos un índice entre imaginación mano derecha e izquierda
- **Prueba:** Mann-Whitney U (T1 vs T2 en runs de imaginación)


## 6. Clasificación

### 6.1 Selección de características

Se seleccionan los índices con mayor diferencia estadística entre condiciones, usando como criterio:

- p-valor < 0.05 en pruebas de hipótesis
- Mayor separabilidad visual en boxplots

### 6.2 Estrategia de entrenamiento — K-Fold Cross Validation

Los enfoques actuales en el diseño de sistemas MI-BCI requieren métodos efectivos de extracción de características y algoritmos de clasificación para adquirir características discriminativas, debido a la naturaleza no lineal y no estacionaria de las señales EEG. El K-Fold garantiza una evaluación robusta e imparcial del clasificador[1].

Se usa K-Fold con k=5 para evaluar el rendimiento de los clasificadores de forma robusta:

```
Dataset completo (109 sujetos)
         ↓
    K-Fold (k=5)
    ┌─────────────────────────────┐
    │ Fold 1: 80% train, 20% test │
    │ Fold 2: 80% train, 20% test │
    │ Fold 3: 80% train, 20% test │
    │ Fold 4: 80% train, 20% test │
    │ Fold 5: 80% train, 20% test │
    └─────────────────────────────┘
         ↓
  Promedio de métricas
  (accuracy, F1-score, AUC-ROC)
```

### 6.3 Arquitecturas de clasificación [11]

| Modelo | Librería | Por qué usarlo |
|---|---|---|
| MLP (Red neuronal) | sklearn / keras | Modelo base de referencia |
| CNN | keras/tensorflow | Captura patrones espaciales en la grilla 3×3 |
| LSTM | keras/tensorflow | Captura dependencias temporales en EEG |
| SVM | sklearn | Robusto con pocas muestras, estándar en BCI |
| XGBoost | xgboost | Alto rendimiento en datos tabulares |

### 6.4 Métricas de evaluación

- **Accuracy** → porcentaje de clasificaciones correctas
- **F1-score** → balance entre precisión y recall
- **Matriz de confusión** → errores por clase
- **AUC-ROC** → capacidad discriminante del modelo

# Proyecto 2 — Clasificación de señales EEG para BCI
## Bioseñales y Sistemas 2026-01

Este proyecto es la continuación del proyecto 1. Se extraen índices adicionales 
a la PSD para mejorar la clasificación de 3 condiciones:
- **Reposo**
- **Imaginación mano derecha**
- **Imaginación mano izquierda**

In [13]:
#1 Importar librerías

import mne                          # Para leer y procesar EEG
import numpy as np                  # Operaciones matemáticas vectorizadas
import pandas as pd                 # Para crear el DataFrame de resultados
import matplotlib.pyplot as plt     # Para graficar
import seaborn as sns               # Para gráficas estadísticas
from scipy import signal            # Para cálculo de PSD y Welch
from scipy import stats             # Para pruebas estadísticas
from pathlib import Path            # Para manejar rutas de archivos
from sklearn.preprocessing import LabelEncoder  # Para codificar etiquetas
from sklearn.model_selection import KFold       # Para validación cruzada

# Configuración general
mne.set_log_level('WARNING')
%matplotlib inline

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [14]:
#2 Rutas y parámetros globales

# Ruta donde están los datos descargados
RUTA_DATOS = Path("./eeg-motor-movementimagery-dataset-1.0.0/files")

# Ruta al CSV del proyecto 1 (ya calculado)
RUTA_CSV_P1 = Path("./resultados_bci.csv")

# 9 canales sobre zona sensoriomotora bilateral
# C3 → hemisferio izquierdo → mano derecha
# C4 → hemisferio derecho → mano izquierda
CANALES_INTERES = [
    'Fc3.', 'Fcz.', 'Fc4.',   # Corteza motora anterior
    'C3..', 'Cz..', 'C4..',   # Corteza motora primaria 
    'Cp3.', 'Cpz.', 'Cp4.'    # Corteza somatosensorial
]

# Bandas de frecuencia
BANDAS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),   # Banda Mu en contexto motor 
    'beta':  (13, 30),  # Control motor 
    'gamma': (30, 40)
}

# Parámetros de adquisición
FS = 160  # Hz

# Filtros
FREQ_MIN = 0.5
FREQ_MAX = 40.0
FREQ_NOTCH = 60

# Condiciones de interés para proyecto 2
# Solo runs de IMAGINACIÓN (R04, R08, R12)
# T0=reposo, T1=mano izquierda, T2=mano derecha
CONDICIONES_P2 = {
    'T0': 'reposo',
    'T1': 'imaginacion_mano_izquierda',
    'T2': 'imaginacion_mano_derecha'
}

# Verificación
print("Parámetros cargados correctamente")
print(f"   Ruta datos: {RUTA_DATOS}")
print(f"   Canales: {CANALES_INTERES}")
print(f"   Bandas: {list(BANDAS.keys())}")
print(f"   Condiciones P2: {CONDICIONES_P2}")

Parámetros cargados correctamente
   Ruta datos: eeg-motor-movementimagery-dataset-1.0.0\files
   Canales: ['Fc3.', 'Fcz.', 'Fc4.', 'C3..', 'Cz..', 'C4..', 'Cp3.', 'Cpz.', 'Cp4.']
   Bandas: ['delta', 'theta', 'alpha', 'beta', 'gamma']
   Condiciones P2: {'T0': 'reposo', 'T1': 'imaginacion_mano_izquierda', 'T2': 'imaginacion_mano_derecha'}


## Pipeline de procesamiento (heredado del proyecto 1)

Se reutilizan las funciones validadas en el proyecto 1:
- `procesar_eeg()` → filtrado pasa banda + Notch + selección de canales
- `es_run_imaginacion()` → filtra solo runs R04, R08, R12

In [15]:
#3 Funciones heredadas del proyecto 1

import re

def procesar_eeg(ruta_archivo, mostrar_pasos=False):
    """
    Aplica el pipeline de procesamiento completo sobre un archivo EDF.
    Heredada del proyecto 1.
    """
    raw = mne.io.read_raw_edf(ruta_archivo, preload=True)
    
    if mostrar_pasos:
        print(f"Archivo cargado: {len(raw.ch_names)} canales, "
              f"{raw.times[-1]:.1f}s, {raw.info['sfreq']} Hz")
    
    # Seleccionar canales sensoriomotores
    canales_disponibles = [c for c in CANALES_INTERES if c in raw.ch_names]
    raw.pick(canales_disponibles)
    
    # Filtro pasa banda FIR
    raw.filter(l_freq=FREQ_MIN, h_freq=FREQ_MAX, method='fir')
    
    # Filtro Notch
    raw.notch_filter(freqs=FREQ_NOTCH)
    
    return raw


def es_run_imaginacion(nombre_archivo):
    """
    Retorna True solo si el archivo es un run de imaginación
    de manos (R04, R08, R12) — únicos runs relevantes para proyecto 2.
    T1 = imaginación mano izquierda
    T2 = imaginación mano derecha
    """
    match = re.search(r'R(\d+)', str(nombre_archivo))
    if match is None:
        return False
    run = int(match.group(1))
    return run in [4, 8, 12]


print("Funciones del proyecto 1 + filtro de runs cargadas correctamente")
print("\n=== PRUEBA es_run_imaginacion ===")
for archivo in ["S001R03.edf", "S001R04.edf", "S001R08.edf", 
                "S001R12.edf", "S001R14.edf"]:
    print(f"   {archivo} → {es_run_imaginacion(archivo)}")

Funciones del proyecto 1 + filtro de runs cargadas correctamente

=== PRUEBA es_run_imaginacion ===
   S001R03.edf → False
   S001R04.edf → True
   S001R08.edf → True
   S001R12.edf → True
   S001R14.edf → False


## Índices adicionales — ERD/ERS, Hjorth y Entropía Espectral

Se implementan 3 índices adicionales a la PSD que mejoran la clasificación 
en sistemas BCI según la literatura consultada.

In [16]:
#4 Funciones para calcular los nuevos índices


def calcular_erd_ers(segmento_tarea, segmento_reposo, fs, banda):
    """
    Calcula ERD/ERS como porcentaje de cambio relativo de potencia
    respecto al reposo para una banda de frecuencia.
    
    Parámetros:
        segmento_tarea:  array (n_canales, n_muestras) — T1 o T2
        segmento_reposo: array (n_canales, n_muestras) — T0
        fs: frecuencia de muestreo (160 Hz)
        banda: tupla (f_min, f_max)
    
    Retorna:
        erd_ers: array (n_canales,) con porcentaje ERD/ERS por canal
    """
    f_min, f_max = banda
    n_canales = segmento_tarea.shape[0]
    erd_ers = np.zeros(n_canales)
    
    for idx in range(n_canales):
        # PSD tarea y reposo con Welch
        freqs, psd_tarea = signal.welch(
            segmento_tarea[idx], fs=fs, nperseg=int(fs*2))
        _, psd_reposo = signal.welch(
            segmento_reposo[idx], fs=fs, nperseg=int(fs*2))
        
        # Índices de la banda
        idx_banda = np.where((freqs >= f_min) & (freqs <= f_max))[0]
        
        # Potencia promedio en la banda
        p_tarea  = np.mean(psd_tarea[idx_banda])
        p_reposo = np.mean(psd_reposo[idx_banda])
        
        # ERD/ERS como porcentaje
        erd_ers[idx] = ((p_tarea - p_reposo) / (p_reposo + 1e-10)) * 100
    
    return erd_ers


def calcular_hjorth(segmento):
    """
    Calcula los 3 parámetros de Hjorth sobre un segmento EEG.
    
    Parámetros:
        segmento: array (n_canales, n_muestras)
    
    Retorna:
        actividad:   array (n_canales,)
        movilidad:   array (n_canales,)
        complejidad: array (n_canales,)
    """
    n_canales = segmento.shape[0]
    actividad   = np.zeros(n_canales)
    movilidad   = np.zeros(n_canales)
    complejidad = np.zeros(n_canales)
    
    for idx in range(n_canales):
        señal = segmento[idx]
        
        # Actividad: varianza de la señal
        actividad[idx] = np.var(señal)
        
        # Primera derivada
        d1 = np.diff(señal)
        
        # Movilidad: frecuencia media
        movilidad[idx] = np.std(d1) / (np.std(señal) + 1e-10)
        
        # Segunda derivada
        d2 = np.diff(d1)
        
        # Complejidad: cambio en la movilidad
        mov_d1 = np.std(d2) / (np.std(d1) + 1e-10)
        complejidad[idx] = mov_d1 / (movilidad[idx] + 1e-10)
    
    return actividad, movilidad, complejidad


def calcular_entropia_espectral(segmento, fs, banda):
    """
    Calcula la entropía espectral de Shannon en una banda de frecuencia.
    
    Parámetros:
        segmento: array (n_canales, n_muestras)
        fs: frecuencia de muestreo (160 Hz)
        banda: tupla (f_min, f_max)
    
    Retorna:
        entropia: array (n_canales,)
    """
    f_min, f_max = banda
    n_canales = segmento.shape[0]
    entropia = np.zeros(n_canales)
    
    for idx in range(n_canales):
        # PSD con Welch
        freqs, psd = signal.welch(
            segmento[idx], fs=fs, nperseg=int(fs*2))
        
        # Filtrar por banda
        idx_banda = np.where((freqs >= f_min) & (freqs <= f_max))[0]
        psd_banda = psd[idx_banda]
        
        # Normalizar para obtener distribución de probabilidad
        psd_norm = psd_banda / (np.sum(psd_banda) + 1e-10)
        
        # Entropía de Shannon
        entropia[idx] = -np.sum(psd_norm * np.log2(psd_norm + 1e-10))
    
    return entropia


print("Funciones de índices adicionales definidas correctamente")
print("   - calcular_erd_ers()")
print("   - calcular_hjorth()")
print("   - calcular_entropia_espectral()")

Funciones de índices adicionales definidas correctamente
   - calcular_erd_ers()
   - calcular_hjorth()
   - calcular_entropia_espectral()


## Función principal de extracción de características
### Punto 1 de programación (20%)

Función que recibe una señal EEG ya procesada y retorna todos los índices 
para los segmentos T0, T1 y T2.

In [17]:
#5 Función principal de extracción de características

def extraer_caracteristicas(raw):
    """
    Recibe una señal EEG ya procesada y extrae todos los índices
    para cada segmento T0, T1, T2:
        - PSD por bandas
        - ERD/ERS (solo T1 y T2 respecto a T0)
        - Parámetros de Hjorth
        - Entropía espectral por banda

    Parámetros:
        raw: objeto MNE con la señal ya procesada

    Retorna:
        resultado: diccionario con todas las características
    """
    fs = raw.info['sfreq']
    resultado = {}

    # --- Paso 1: Agrupar segmentos por etiqueta ---
    segmentos_por_etiqueta = {'T0': [], 'T1': [], 'T2': []}

    for ann in raw.annotations:
        etiqueta = ann['description']
        if etiqueta not in segmentos_por_etiqueta:
            continue
        inicio = int(ann['onset'] * fs)
        fin = inicio + int(ann['duration'] * fs)
        datos_segmento = raw.get_data()[:, inicio:fin]
        segmentos_por_etiqueta[etiqueta].append(datos_segmento)

    # Verificar que T0 existe
    if len(segmentos_por_etiqueta['T0']) == 0:
        return resultado

    #  CORRECCIÓN: truncar todos los segmentos al mínimo tamaño
    # para poder apilarlos aunque tengan diferente duración
    def promediar_segmentos(segmentos):
        min_muestras = min(s.shape[1] for s in segmentos)
        segmentos_truncados = [s[:, :min_muestras] for s in segmentos]
        return np.mean(segmentos_truncados, axis=0)

    # Promediar segmentos de reposo (T0) — línea base para ERD/ERS
    segmento_reposo = promediar_segmentos(segmentos_por_etiqueta['T0'])

    # --- Paso 2: Calcular características por etiqueta ---
    for etiqueta, segmentos in segmentos_por_etiqueta.items():
        if len(segmentos) == 0:
            continue

        # Promediar segmentos del mismo tipo
        segmento = promediar_segmentos(segmentos)

        for i, canal in enumerate(raw.ch_names):

            señal = segmento[i]
            señal_reposo = segmento_reposo[i]

            # --- PSD por bandas ---
            freqs, psd = signal.welch(señal, fs=fs, nperseg=int(fs*2))
            _, psd_r = signal.welch(señal_reposo, fs=fs, nperseg=int(fs*2))

            for nombre_banda, (f_min, f_max) in BANDAS.items():
                idx_b = np.where((freqs >= f_min) & (freqs <= f_max))[0]

                # PSD
                potencia = np.mean(psd[idx_b])
                resultado[f'{etiqueta}_psd_{nombre_banda}_{canal}'] = potencia

                # Entropía espectral
                psd_banda = psd[idx_b]
                psd_norm = psd_banda / (np.sum(psd_banda) + 1e-10)
                entropia = -np.sum(psd_norm * np.log2(psd_norm + 1e-10))
                resultado[f'{etiqueta}_entropia_{nombre_banda}_{canal}'] = entropia

                # ERD/ERS (solo T1 y T2)
                if etiqueta != 'T0':
                    p_tarea = np.mean(psd[idx_b])
                    p_reposo = np.mean(psd_r[idx_b])
                    erd = ((p_tarea - p_reposo) / (p_reposo + 1e-10)) * 100
                    resultado[f'{etiqueta}_erd_{nombre_banda}_{canal}'] = erd

            # --- Parámetros de Hjorth ---
            actividad = np.var(señal)
            d1 = np.diff(señal)
            movilidad = np.std(d1) / (np.std(señal) + 1e-10)
            d2 = np.diff(d1)
            mov_d1 = np.std(d2) / (np.std(d1) + 1e-10)
            complejidad = mov_d1 / (movilidad + 1e-10)

            resultado[f'{etiqueta}_hjorth_actividad_{canal}'] = actividad
            resultado[f'{etiqueta}_hjorth_movilidad_{canal}'] = movilidad
            resultado[f'{etiqueta}_hjorth_complejidad_{canal}'] = complejidad

    return resultado

print("Función extraer_caracteristicas() corregida correctamente")

Función extraer_caracteristicas() corregida correctamente


In [18]:
#6 Prueba de extraer_caracteristicas()

# Probamos con un archivo de imaginación (R04)
archivo_prueba = RUTA_DATOS / "S001" / "S001R04.edf"

# Procesar la señal
raw_prueba = procesar_eeg(archivo_prueba, mostrar_pasos=False)

# Extraer características
caracteristicas_prueba = extraer_caracteristicas(raw_prueba)

# Mostrar resumen
print(f"Características extraídas: {len(caracteristicas_prueba)} valores")
print(f"\n=== TIPOS DE CLAVES ===")

# Mostrar un ejemplo de cada índice
ejemplos = ['T0_psd_alpha_C3..', 'T1_psd_alpha_C3..', 'T2_psd_alpha_C3..',
            'T1_erd_alpha_C3..', 'T2_erd_alpha_C3..',
            'T1_hjorth_actividad_C3..', 'T1_hjorth_movilidad_C3..', 
            'T1_hjorth_complejidad_C3..',
            'T1_entropia_alpha_C3..', 'T2_entropia_beta_C4..']

for clave in ejemplos:
    if clave in caracteristicas_prueba:
        print(f"   {clave}: {caracteristicas_prueba[clave]:.6f}")

# Verificar estructura
print(f"\n=== VERIFICACIÓN ===")
claves_psd = [c for c in caracteristicas_prueba if '_psd_' in c]
claves_erd = [c for c in caracteristicas_prueba if '_erd_' in c]
claves_hjorth = [c for c in caracteristicas_prueba if '_hjorth_' in c]
claves_entropia = [c for c in caracteristicas_prueba if '_entropia_' in c]

print(f"   PSD:      {len(claves_psd)} valores (esperado: {3*5*9} = 135)")
print(f"   ERD/ERS:  {len(claves_erd)} valores (esperado: {2*5*9} = 90)")
print(f"   Hjorth:   {len(claves_hjorth)} valores (esperado: {3*3*9} = 81)")
print(f"   Entropía: {len(claves_entropia)} valores (esperado: {3*5*9} = 135)")
print(f"   TOTAL:    {len(caracteristicas_prueba)} valores (esperado: 441)")

Características extraídas: 441 valores

=== TIPOS DE CLAVES ===
   T0_psd_alpha_C3..: 0.000000
   T1_psd_alpha_C3..: 0.000000
   T2_psd_alpha_C3..: 0.000000
   T1_erd_alpha_C3..: 1.242971
   T2_erd_alpha_C3..: 1.547070
   T1_hjorth_actividad_C3..: 0.000000
   T1_hjorth_movilidad_C3..: 0.336692
   T1_hjorth_complejidad_C3..: 2.784629
   T1_entropia_alpha_C3..: 1.375523
   T2_entropia_beta_C4..: 1.735876

=== VERIFICACIÓN ===
   PSD:      135 valores (esperado: 135 = 135)
   ERD/ERS:  90 valores (esperado: 90 = 90)
   Hjorth:   81 valores (esperado: 81 = 81)
   Entropía: 135 valores (esperado: 135 = 135)
   TOTAL:    441 valores (esperado: 441)


## Rutina principal — Procesamiento masivo
### Punto 2 de programación (10%)

Aplica el pipeline completo sobre todos los archivos de imaginación 
(R04, R08, R12) y almacena resultados en un DataFrame con:
- sujeto, archivo, run
- 441 características por registro

In [19]:
#7 Rutina principal - procesar todos los archivos

def procesar_dataset_p2(ruta_datos, max_sujetos=None):
    """
    Aplica el pipeline completo sobre todos los archivos de imaginación
    (R04, R08, R12) del dataset y almacena resultados en un DataFrame.

    Parámetros:
        ruta_datos: Path a la carpeta con los datos
        max_sujetos: número máximo de sujetos (None = todos)

    Retorna:
        df: DataFrame con sujeto, condicion y todas las métricas
    """

    # Buscar todas las carpetas de sujetos
    sujetos = sorted([s for s in ruta_datos.iterdir() if s.is_dir()])

    if max_sujetos is not None:
        sujetos = sujetos[:max_sujetos]

    print(f"Sujetos a procesar: {len(sujetos)}")

    filas = []

    for i, carpeta_sujeto in enumerate(sujetos):

        sujeto = carpeta_sujeto.name
        print(f"\nSujeto {i+1}/{len(sujetos)}: {sujeto}")

        # Filtrar solo archivos de imaginación de manos
        archivos = sorted([
            f for f in carpeta_sujeto.glob("*.edf")
            if es_run_imaginacion(f.name)
        ])

        for archivo in archivos:
            print(f"   {archivo.name}")

            try:
                # Paso 1: procesar señal EEG
                raw = procesar_eeg(archivo, mostrar_pasos=False)

                # Paso 2: extraer todas las características
                caracteristicas = extraer_caracteristicas(raw)

                # Paso 3: construir fila del DataFrame
                fila = {
                    'sujeto':   sujeto,
                    'archivo':  archivo.name,
                    'run':      archivo.name.split('R')[1].split('.')[0],
                }
                fila.update(caracteristicas)
                filas.append(fila)

            except Exception as e:
                print(f"   Error: {e}")
                continue

    df = pd.DataFrame(filas)

    print(f"\nProcesamiento completado!")
    print(f"   Total registros: {len(df)}")
    print(f"   Columnas: {len(df.columns)}")

    return df

print(" Función procesar_dataset_p2() definida correctamente")

 Función procesar_dataset_p2() definida correctamente


In [20]:
#8 Prueba con 2 sujetos

# Probamos con 2 sujetos para verificar que funciona
df_prueba = procesar_dataset_p2(RUTA_DATOS, max_sujetos=2)

# Ver estructura
print("\n=== PRIMERAS FILAS ===")
print(df_prueba[['sujeto', 'archivo', 'run']].to_string())

print(f"\n=== COLUMNAS ({len(df_prueba.columns)}) ===")
print(f"   Identificadores: sujeto, archivo, run")
print(f"   PSD: {len([c for c in df_prueba.columns if '_psd_' in c])} columnas")
print(f"   ERD/ERS: {len([c for c in df_prueba.columns if '_erd_' in c])} columnas")
print(f"   Hjorth: {len([c for c in df_prueba.columns if '_hjorth_' in c])} columnas")
print(f"   Entropía: {len([c for c in df_prueba.columns if '_entropia_' in c])} columnas")

Sujetos a procesar: 2

Sujeto 1/2: S001
   S001R04.edf
   S001R08.edf
   S001R12.edf

Sujeto 2/2: S002
   S002R04.edf
   S002R08.edf
   S002R12.edf

Procesamiento completado!
   Total registros: 6
   Columnas: 444

=== PRIMERAS FILAS ===
  sujeto      archivo run
0   S001  S001R04.edf  04
1   S001  S001R08.edf  08
2   S001  S001R12.edf  12
3   S002  S002R04.edf  04
4   S002  S002R08.edf  08
5   S002  S002R12.edf  12

=== COLUMNAS (444) ===
   Identificadores: sujeto, archivo, run
   PSD: 135 columnas
   ERD/ERS: 90 columnas
   Hjorth: 81 columnas
   Entropía: 135 columnas


In [21]:
#9 Ejecutar procesamiento completo

df_p2 = procesar_dataset_p2(RUTA_DATOS, max_sujetos=None)

# Guardar resultados
df_p2.to_csv("resultados_p2.csv", index=False)
print("\n Resultados guardados en resultados_p2.csv")

# Vista previa
df_p2.head()

Sujetos a procesar: 109

Sujeto 1/109: S001
   S001R04.edf
   S001R08.edf
   S001R12.edf

Sujeto 2/109: S002
   S002R04.edf
   S002R08.edf
   S002R12.edf

Sujeto 3/109: S003
   S003R04.edf
   S003R08.edf
   S003R12.edf

Sujeto 4/109: S004
   S004R04.edf
   S004R08.edf
   S004R12.edf

Sujeto 5/109: S005
   S005R04.edf
   S005R08.edf
   S005R12.edf

Sujeto 6/109: S006
   S006R04.edf
   S006R08.edf
   S006R12.edf

Sujeto 7/109: S007
   S007R04.edf
   S007R08.edf
   S007R12.edf

Sujeto 8/109: S008
   S008R04.edf
   S008R08.edf
   S008R12.edf

Sujeto 9/109: S009
   S009R04.edf
   S009R08.edf
   S009R12.edf

Sujeto 10/109: S010
   S010R04.edf
   S010R08.edf
   S010R12.edf

Sujeto 11/109: S011
   S011R04.edf
   S011R08.edf
   S011R12.edf

Sujeto 12/109: S012
   S012R04.edf
   S012R08.edf
   S012R12.edf

Sujeto 13/109: S013
   S013R04.edf
   S013R08.edf
   S013R12.edf

Sujeto 14/109: S014
   S014R04.edf
   S014R08.edf
   S014R12.edf

Sujeto 15/109: S015
   S015R04.edf
   S015R08.edf
   S015R12

c:\Users\Valentina Garcia\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 176, using nperseg = 176
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
c:\Users\Valentina Garcia\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 176, using nperseg = 176
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
c:\Users\Valentina Garcia\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 176, using nperseg = 176
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


   S088R12.edf

Sujeto 89/109: S089
   S089R04.edf
   S089R08.edf
   S089R12.edf

Sujeto 90/109: S090
   S090R04.edf
   S090R08.edf
   S090R12.edf

Sujeto 91/109: S091
   S091R04.edf
   S091R08.edf
   S091R12.edf

Sujeto 92/109: S092
   S092R04.edf
   S092R08.edf


c:\Users\Valentina Garcia\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 176, using nperseg = 176
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
c:\Users\Valentina Garcia\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 176, using nperseg = 176
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
c:\Users\Valentina Garcia\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 176, using nperseg = 176
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


   S092R12.edf

Sujeto 93/109: S093
   S093R04.edf
   S093R08.edf
   S093R12.edf

Sujeto 94/109: S094
   S094R04.edf
   S094R08.edf
   S094R12.edf

Sujeto 95/109: S095
   S095R04.edf
   S095R08.edf
   S095R12.edf

Sujeto 96/109: S096
   S096R04.edf
   S096R08.edf
   S096R12.edf

Sujeto 97/109: S097
   S097R04.edf
   S097R08.edf
   S097R12.edf

Sujeto 98/109: S098
   S098R04.edf
   S098R08.edf
   S098R12.edf

Sujeto 99/109: S099
   S099R04.edf
   S099R08.edf
   S099R12.edf

Sujeto 100/109: S100
   S100R04.edf
   S100R08.edf


C:\Users\Valentina Garcia\AppData\Local\Temp\ipykernel_38328\1275428672.py:10: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(ruta_archivo, preload=True)
C:\Users\Valentina Garcia\AppData\Local\Temp\ipykernel_38328\1275428672.py:10: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(ruta_archivo, preload=True)
C:\Users\Valentina Garcia\AppData\Local\Temp\ipykernel_38328\1275428672.py:10: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(ruta_archivo, preload=True)


   S100R12.edf

Sujeto 101/109: S101
   S101R04.edf
   S101R08.edf
   S101R12.edf

Sujeto 102/109: S102
   S102R04.edf
   S102R08.edf
   S102R12.edf

Sujeto 103/109: S103
   S103R04.edf
   S103R08.edf
   S103R12.edf

Sujeto 104/109: S104
   S104R04.edf
   S104R08.edf
   S104R12.edf

Sujeto 105/109: S105
   S105R04.edf
   S105R08.edf
   S105R12.edf

Sujeto 106/109: S106
   S106R04.edf
   S106R08.edf
   S106R12.edf

Sujeto 107/109: S107
   S107R04.edf
   S107R08.edf
   S107R12.edf

Sujeto 108/109: S108
   S108R04.edf
   S108R08.edf
   S108R12.edf

Sujeto 109/109: S109
   S109R04.edf
   S109R08.edf
   S109R12.edf

Procesamiento completado!
   Total registros: 327
   Columnas: 444

 Resultados guardados en resultados_p2.csv


,sujeto,archivo,run,T0_psd_delta_Fc3.,T0_entropia_delta_Fc3.,T0_psd_theta_Fc3.,T0_entropia_theta_Fc3.,T0_psd_alpha_Fc3.,T0_entropia_alpha_Fc3.,T0_psd_beta_Fc3.,...,T2_erd_alpha_Cp4.,T2_psd_beta_Cp4.,T2_entropia_beta_Cp4.,T2_erd_beta_Cp4.,T2_psd_gamma_Cp4.,T2_entropia_gamma_Cp4.,T2_erd_gamma_Cp4.,T2_hjorth_actividad_Cp4.,T2_hjorth_movilidad_Cp4.,T2_hjorth_complejidad_Cp4.
0,S001,S001R04.edf,04,1.925213e-11,2.231497,5.003767e-12,1.434148,1.974948e-12,1.034574,8.096866e-13,...,3.232180,9.981500e-13,1.735070,0.299018,3.157274e-13,0.502919,0.097711,3.168674e-10,0.283913,3.385449
1,S001,S001R08.edf,08,2.050543e-11,2.242556,5.888526e-12,1.529707,2.074821e-12,1.062006,7.955199e-13,...,1.331687,1.360244e-12,1.977332,0.771980,2.466678e-13,0.420834,0.136436,2.719962e-10,0.292425,3.038600
2,S001,S001R12.edf,12,2.306789e-11,2.173732,4.185230e-12,1.301874,2.545587e-12,1.204442,7.534838e-13,...,0.956018,8.976648e-13,1.646386,0.457301,1.976022e-13,0.349374,0.093242,2.481255e-10,0.287862,3.117256
3,S002,S002R04.edf,04,2.679945e-12,0.930347,8.157441e-13,0.450670,5.689916e-13,0.433784,2.772785e-13,...,0.574413,4.102475e-13,0.979460,0.199324,1.059004e-13,0.208901,0.029860,4.163848e-11,0.479720,2.026103
4,S002,S002R08.edf,08,3.617070e-12,1.109868,8.941821e-13,0.482174,8.909276e-13,0.600471,2.160943e-13,...,0.864153,6.876498e-13,1.371435,0.435711,1.119408e-13,0.221828,0.052161,4.807169e-11,0.479792,1.960128


## Carga y preparación de datos para análisis

Se carga el CSV generado y se reorganiza el DataFrame para tener 
una fila por condición (T0=reposo, T1=mano izq, T2=mano der).

In [22]:
#10 Cargar, verificar y reorganizar datos para análisis

# Cargar CSV guardado anteriormente
df_p2 = pd.read_csv("resultados_p2.csv")

print(f"Datos cargados correctamente")
print(f"   Registros: {len(df_p2)}")
print(f"   Columnas: {len(df_p2.columns)}")
print(f"   Sujetos: {df_p2['sujeto'].nunique()}")
print(f"   Registros esperados: 109 × 3 = 327")
print(f"   Registros faltantes: {327 - len(df_p2)}")
print(f"   Valores nulos: {df_p2.isnull().sum().sum()}")

# Reorganizar para tener una fila por CONDICIÓN
# El CSV tiene una fila por archivo con T0, T1, T2 mezclados
# Necesitamos separar:
#   T0 → reposo
#   T1 → imaginacion_mano_izquierda
#   T2 → imaginacion_mano_derecha
filas_reorganizadas = []

for _, fila in df_p2.iterrows():
    for etiqueta, condicion in CONDICIONES_P2.items():
        
        nueva_fila = {
            'sujeto':    fila['sujeto'],
            'archivo':   fila['archivo'],
            'run':       fila['run'],
            'condicion': condicion,
            'etiqueta':  etiqueta
        }
        
        # Extraer características de esta etiqueta
        # y renombrar quitando el prefijo T0_, T1_, T2_
        for col in df_p2.columns:
            if col.startswith(f'{etiqueta}_'):
                nueva_col = col[3:]
                nueva_fila[nueva_col] = fila[col]
        
        filas_reorganizadas.append(nueva_fila)

df_analisis = pd.DataFrame(filas_reorganizadas)

print(f"\nDataFrame reorganizado por condición")
print(f"   Registros: {len(df_analisis)}")
print(f"   Columnas: {len(df_analisis.columns)}")
print(f"\n=== CONDICIONES ===")
print(df_analisis['condicion'].value_counts())

Datos cargados correctamente
   Registros: 327
   Columnas: 444
   Sujetos: 109
   Registros esperados: 109 × 3 = 327
   Registros faltantes: 0
   Valores nulos: 0

DataFrame reorganizado por condición
   Registros: 981
   Columnas: 167

=== CONDICIONES ===
condicion
reposo                        327
imaginacion_mano_izquierda    327
imaginacion_mano_derecha      327
Name: count, dtype: int64


In [23]:
#11 Preparar datos para graficar

# Convertir PSD de V²/Hz a µV²/Hz (×1e12) para mejor visualización
df_plot = df_analisis.copy()

cols_psd = [c for c in df_plot.columns if c.startswith('psd_')]
for col in cols_psd:
    df_plot[col] = df_plot[col] * 1e12

# Eliminar outliers extremos en PSD (percentil 95)
# Justificación: reducir efecto de artefactos en visualización
for col in cols_psd:
    p95 = df_plot[col].quantile(0.95)
    df_plot[col] = df_plot[col].clip(upper=p95)

print("Datos preparados para graficar")
print(f"   Columnas PSD convertidas a µV²/Hz: {len(cols_psd)}")
print(f"\n=== EJEMPLO VALORES POR CONDICIÓN ===")
print(f"\nPSD alpha C3 (µV²/Hz):")
print(df_plot.groupby('condicion')['psd_alpha_C3..'].mean().round(4))
print(f"\nERD/ERS alpha C3 (%):")
erd_cols = [c for c in df_plot.columns if 'erd_alpha_C3' in c]
if erd_cols:
    print(df_plot[df_plot['condicion']!='reposo'].groupby('condicion')[erd_cols[0]].mean().round(4))
print(f"\nHjorth movilidad C3:")
print(df_plot.groupby('condicion')['hjorth_movilidad_C3..'].mean().round(4))

Datos preparados para graficar
   Columnas PSD convertidas a µV²/Hz: 45

=== EJEMPLO VALORES POR CONDICIÓN ===

PSD alpha C3 (µV²/Hz):
condicion
imaginacion_mano_derecha      4.2543
imaginacion_mano_izquierda    4.1909
reposo                        2.7339
Name: psd_alpha_C3.., dtype: float64

ERD/ERS alpha C3 (%):
condicion
imaginacion_mano_derecha      1.6612
imaginacion_mano_izquierda    1.7700
Name: erd_alpha_C3.., dtype: float64

Hjorth movilidad C3:
condicion
imaginacion_mano_derecha      0.3469
imaginacion_mano_izquierda    0.3526
reposo                        0.3318
Name: hjorth_movilidad_C3.., dtype: float64


## Referencias

[1]Oh, S.-H., Lee, Y.-R., & Kim, H.-N. (2014). A novel EEG feature extraction method using Hjorth parameter. International Journal of Electronics and Electrical Engineering, 2(2), 106–110. https://doi.org/10.12720/ijeee.2.2.106-110

[2]Zhang, S., Wang, S., Zheng, D., Zhu, K., & Dai, M. (2019). A novel pattern with high-level commands for encoding motor imagery-based brain computer interface. Pattern Recognition Letters, 125, 574–580. https://doi.org/10.1016/j.patrec.2019.03.020

[3]Lotte, F., Van Langhenhove, A., Lamarche, F., Ernest, T., Renard, Y., Arnaldi, B., & Lécuyer, A. (2010). Exploring large virtual environments by thoughts using a brain–computer interface based on motor imagery and high-level commands. Presence: Teleoperators and Virtual Environments, 19(1), 54–70. https://doi.org/10.1162/pres.19.1.54

[4]Bodda, S., & Diwakar, S. (2022). Exploring EEG spectral and temporal dynamics underlying a hand grasp movement. PLOS ONE, 17(6), e0270366. https://doi.org/10.1371/journal.pone.0270366

[5]Kauati-Saito, E., Pereira, A. da S., Fontana, A. P., de Sá, A. M. F. L. M., Soares, J. G. M., & Tierra-Criollo, C. J. (2025). Classification of different motor imagery tasks with the same limb using electroencephalographic signals. Sensors, 25(17), 5291. https://doi.org/10.3390/s25175291

[6]Wang, Y., Gao, X., Hong, B., & Gao, S. (2010). Practical designs of brain-computer interfaces based on the modulation of EEG rhythms. En B. Graimann, G. Pfurtscheller, & B. Allison (Eds.), Brain-computer interfaces: Revolutionizing human-computer interaction (pp. 137–154). Springer. https://doi.org/10.1007/978-3-642-02091-9_8

[7]Degirmenci, M., Yuce, Y. K., Perc, M., & Isler, Y. (2024). EEG channel and feature investigation in binary and multiple motor imagery task predictions. Frontiers in Human Neuroscience, 18, 1525139. https://doi.org/10.3389/fnhum.2024.1525139

[8]Kitahara, K., Hayashi, Y., Yano, S., & Kondo, T. (2017). Target-directed motor imagery of the lower limb enhances event-related desynchronization. PLOS ONE, 12(9), e0184245. https://doi.org/10.1371/journal.pone.0184245

[9]Yakovlev, L., Syrov, N., Miroshnikov, A., Lebedev, M., & Kaplan, A. (2023). Event-related desynchronization induced by tactile imagery: an EEG study. eNeuro, 10(6), ENEURO.0455-22.2023. https://doi.org/10.1523/ENEURO.0455-22.2023

[10]Prasad, A. (2016). Feature extraction and classification for motor imagery in EEG signals [Proyecto final de Maestría, Kaunas University of Technology]. Kaunas University of Technology.

[11]Kaushik, G., Gaur, P., Sharma, R. R., & Pachori, R. B. (2022). EEG signal based seizure detection focused on Hjorth parameters from tunable-Q wavelet sub-bands. Biomedical Signal Processing and Control, 76, 103645. https://doi.org/10.1016/j.bspc.2022.103645